# 08 · RAG 检索问答 —— 开卷考，先翻资料再答题

**家族位置**：08 生产级优化第 8 站。07 是扩参数装知识，本章是外挂知识不扩参数：12 篇中文小文档 + TF-IDF 检索 + 抽取式问答 + 库外拒答。

**学习目标**：建库→检索→抽取→拒答全链路；recall@1/@k + 抽取命中率；TopK/阈值消融；与 MoE 扩参数对照。

## 1. 原理：分诊台升级成图书馆

### 通俗理解

**一句话**：闭卷考靠背（参数记知识，07-MoE 是扩招背书团）；开卷考靠翻（RAG 是建图书馆，问题来了先检索再摘句）——知识更新只换书，不用重训人。

### 结构账

```
语料： 12 篇中文（RoPE/KV/GQA/Flash/LoRA/AMP/ckpt/量化/剪枝/蒸馏/MoE/RAG 各 1 篇，2-3 句）
检索： jieba 分词 + 去停用词 → TF-IDF → 余弦 TopK（阈值 0.08 拒答）
问答： 最佳文档内按 query 词重叠抽最佳句；低分/库外问题拒答
评估： 24 问（12 库内 + 12 换述/库外），recall@1/@3 + 抽取命中率 + 拒答率
```

In [ ]:
import sys
from pathlib import Path
import numpy as np
import matplotlib.pyplot as plt
ROOT=Path.cwd()
while ROOT != ROOT.parent and not (ROOT/'common').exists(): ROOT=ROOT.parent
sys.path.insert(0,str(ROOT))
from common.rag import ZhRAG, tokenize_zh
from common.utils import set_seed,setup_chinese_font
set_seed(0); setup_chinese_font()
FIGS=Path.cwd()/'figs'; FIGS.mkdir(exist_ok=True)
docs=[
 'RoPE 位置编码把相对位移编进 Q/K 的旋转矩阵，点积只依赖 m-n，外推长度时无需新位置表。',
 'KV Cache 在自回归推理时缓存历史 K/V，每步只算新 token，把 O(S²) 重算降到 O(S)。',
 'GQA 把 kv 头从 H 压到 G（如 8），KV 显存按 H/G 倍降，是 MHA 与 MQA 的折中。',
 'FlashAttention 用 online softmax 分块归约，S×S 中间矩阵永不物化，HBM 访问量大降。',
 'LoRA 冻结基座权重，只训练低秩增量 B·A（r=8），推理可 merge 回 W 零开销。',
 'AMP 混合精度用 bf16 粗算、fp32 保 loss 与权重，精度不掉且省显存。',
 '梯度检查点前向不存激活、反向重算，用一次前向时间换 O(√K) 内存。',
 'INT8 PTQ 用 scale=max|w|/127 对称量化，体积÷4，MNIST 上精度几乎不掉。',
 '幅度剪枝把最小 |w| 的 s% 置零，s=0.7 前精度不掉，s=0.9 崩。',
 '知识蒸馏用 T=4 软标签 + T²·KL 让学生学类间关系，学生欠拟合时最赚。',
 'MoE 用门控 8 选 top-2 专家，aux loss 保负载均衡，总参 3× 但激活恒定。',
 'RAG 先检索再生成，知识更新只换文档不重训，是本章自己的定义。',
]
print(f'语料 {len(docs)} 篇 | 示例分词: {tokenize_zh(docs[0])[:40]}...')
rag=ZhRAG(docs,threshold=0.08)
print('建库 OK：TF-IDF 矩阵',tuple(rag.mat.shape))

## 2. 问答演示：库内 4 问 + 库外 2 问

In [ ]:
demos=['LoRA 的低秩增量是什么？','KV Cache 把复杂度降到多少？','MoE 选几个专家？','剪枝多少会崩？','今天天气怎么样？','谁拿了世界杯冠军？']
print('retrieve Top3:', rag.retrieve('LoRA 的低秩增量是什么？', top_k=3), flush=True)
for q in demos:
    a=rag.answer(q)
    print(f'Q: {q}\nA: {a["text"]}（doc{a["doc_id"]} s={a["score"]:.3f}）',flush=True)
fig,ax=plt.subplots(figsize=(6,3.0)); ax.axis('off')
for i,q in enumerate(demos):
    a=rag.answer(q)
    ax.text(0.02,0.85-i*0.14,f'Q{i+1}:{q[:12]} → doc{a["doc_id"]} s={a["score"]:.2f} {"OK" if a["known"] else "拒答"}',fontsize=10)
ax.set_title('问答演示：库内摘句，库外拒答')
plt.tight_layout(); plt.savefig(FIGS/'fig1_demo.png',dpi=150,bbox_inches='tight'); plt.show()

## 3. 评估：24 问 recall + 抽取命中 + TopK/阈值消融

In [ ]:
pairs=[('RoPE 相对位移怎么编？',0),('RoPE 点积依赖什么？',0),('KV Cache 缓存什么？',1),('O(S²) 降到多少？',1),
 ('GQA 压到几个头？',2),('MHA 与 MQA 的折中是？',2),('online softmax 有什么用？',3),('S×S 物化吗？',3),
 ('B·A 是什么？',4),('merge 回 W 开销？',4),('bf16 粗算保什么？',5),('AMP 精度掉吗？',5),
 ('反向重算换什么？',6),('O(√K) 内存是谁？',6),('scale 怎么算？',7),('体积除以几？',7),
 ('s=0.9 会怎样？',8),('幅度剪枝置零什么？',8),('T=4 有什么用？',9),('学生何时最赚？',9),
 ('8 选几个？',10),('aux loss 保什么？',10),('先检索再生成是谁？',11),('知识更新要重训吗？',11)]
m=rag.evaluate(pairs,top_k=3)
print(f'24问 recall@1={m["recall@1"]:.4f} recall@3={m["recall@3"]:.4f} 抽取命中={m["answer_hit"]:.4f}',flush=True)
ks=[1,2,3,5]; r1=[rag.evaluate(pairs,top_k=k)['recall@1'] for k in ks]
rk=[rag.evaluate(pairs,top_k=k)['recall@'+str(k)] for k in ks]
ths=[0.0,0.05,0.08,0.15,0.30]; rej=[]
for th in ths:
    r2=ZhRAG(rag.docs,threshold=th)
    outsiders=['今天天气怎么样？','谁拿了世界杯冠军？','股票明天涨吗？','午饭吃什么？']
    rej.append(sum(1 for q in outsiders if not r2.answer(q)['known'])/len(outsiders))
print('TopK→recall:',list(zip(ks,rk)),flush=True)
print('阈值→拒答率:',list(zip(ths,rej)),flush=True)
fig,ax=plt.subplots(1,3,figsize=(11,3.2))
ax[0].bar(['recall@1','recall@3','抽取命中'],[m['recall@1'],m['recall@3'],m['answer_hit']],color=['#4C72B0','#55A868','#DD8452'])
for i,v in enumerate([m['recall@1'],m['recall@3'],m['answer_hit']]): ax[0].text(i,v+0.01,f'{v:.3f}',ha='center')
ax[0].set_ylim(0,1.05); ax[0].set_title('24 问评估')
ax[1].plot(ks,rk,marker='o',color='#4C72B0'); ax[1].set_xlabel('TopK'); ax[1].set_title('TopK 消融')
ax[2].plot(ths,rej,marker='o',color='#DD8452'); ax[2].set_xlabel('阈值'); ax[2].set_title('阈值→库外拒答率')
plt.tight_layout(); plt.savefig(FIGS/'fig2_eval.png',dpi=150,bbox_inches='tight'); plt.show()
print('SUMMARY',round(m['recall@1'],4),round(m['recall@3'],4),round(m['answer_hit'],4),rk,rej)

## 4. 总结与下一步

RAG 闭环：建库→检索→抽取→拒答；24 问 recall@1/@3 + 抽取命中 + 阈值拒答曲线。下一步 `09_Inference_Deployment`：PagedAttention + 投机解码 + GGUF（08 家族收官）。

## 附：TF-IDF 权重可视化（Top 词条）


In [ ]:
import numpy as np
terms=np.array(rag.vec.get_feature_names_out())
weights=np.asarray(rag.mat.sum(axis=0)).flatten()
top=np.argsort(-weights)[:15]
fig,ax=plt.subplots(figsize=(7,3.4))
ax.barh(terms[top][::-1],weights[top][::-1],color='#4C72B0')
ax.set_title('TF-IDF 全库权重 Top15（专家词天然高权）')
plt.tight_layout(); plt.savefig(FIGS/'fig3_tfidf.png',dpi=150,bbox_inches='tight'); plt.show()
print('top terms:',terms[top[:8]].tolist())